# Task 1 — Territorial Digital Divide: Geospatial Raster Analysis

**Region:** Cusco, Peru  
**Datasets:** NASA Black Marble VNL 2025 × OSIPTEL Mobile Coverage 2019  
**Goal:** Measure the territorial digital divide by cross-referencing nighttime lights and mobile connectivity.

---
## Step 0 — Environment Setup

Import required libraries and print versions to confirm the environment is reproducible.

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
import scipy
from scipy import stats
from scipy.ndimage import gaussian_filter
import seaborn as sns
import pandas as pd
import rasterio
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling

print('Library versions:')
print(f'  rasterio:   {rasterio.__version__}')
print(f'  numpy:      {np.__version__}')
print(f'  matplotlib: {matplotlib.__version__}')
print(f'  scipy:      {scipy.__version__}')
print(f'  seaborn:    {sns.__version__}')
print(f'  pandas:     {pd.__version__}')

os.makedirs('../output', exist_ok=True)
print('\nOutput directory ready.')

---
## Step 1 — Raster Loading and Inspection

Load both rasters and print CRS, shape, band count, NoData value, data type, bounding box, pixel resolution, valid pixel count, and value range.

In [ ]:
VNL_PATH  = '../data/VNL_cusco_2025.tif'
CONN_PATH = '../data/kernel_cobmovil2019_50m.tif'

def inspect_raster(path, label):
    with rasterio.open(path) as src:
        data    = src.read(1).astype(np.float64)
        nodata  = src.nodata
        res_deg = src.res                          # (row_res, col_res) in CRS units

        # Valid pixel mask
        if nodata is not None:
            valid_mask = (data != nodata) & ~np.isnan(data)
        else:
            valid_mask = ~np.isnan(data)
        valid_data = data[valid_mask]

        # Approximate km per degree (rough, valid for Cusco latitude ~-13°)
        lat_km = 111.0   # 1° latitude  ≈ 111 km
        lon_km = 111.0 * np.cos(np.radians(13))  # at -13° lat

        print(f"\n{'='*52}")
        print(f"  Raster : {label}")
        print(f"  CRS    : {src.crs}")
        print(f"  Shape  : {src.height} rows × {src.width} cols")
        print(f"  Bands  : {src.count}")
        print(f"  NoData : {nodata}")
        print(f"  Dtype  : {src.dtypes[0]}")
        print(f"  Bounds : {src.bounds}")

        if src.crs and src.crs.is_geographic:
            print(f"  Res    : {res_deg[0]:.6f}° × {res_deg[1]:.6f}°")
            print(f"  Approx : {res_deg[0]*lat_km:.3f} km × {res_deg[1]*lon_km:.3f} km")
        else:
            print(f"  Res    : {res_deg[0]:.2f} m × {res_deg[1]:.2f} m (projected)")

        print(f"  Valid  : {valid_mask.sum():,} px / {data.size:,} total")
        print(f"  Range  : [{valid_data.min():.6f}, {valid_data.max():.6f}]")

        return data, src.meta.copy(), nodata

vnl_raw,  vnl_meta,  vnl_nodata  = inspect_raster(VNL_PATH,  'NASA VNL 2025')
conn_raw, conn_meta, conn_nodata = inspect_raster(CONN_PATH, 'OSIPTEL Mobile Coverage 2019')

---
## Step 2 — Reprojection and Grid Alignment

Two-stage process:
1. Reproject connectivity from **EPSG:32719** → **EPSG:4326** (bilinear resampling).
2. Resample the reprojected layer to the **exact grid** of the VNL raster (same transform and shape).

In [ ]:
dst_crs = CRS.from_epsg(4326)

# --- Read VNL grid parameters (reference grid) ---
with rasterio.open(VNL_PATH) as vnl_src:
    vnl_transform = vnl_src.transform
    vnl_crs       = vnl_src.crs
    vnl_height    = vnl_src.height
    vnl_width     = vnl_src.width

# --- Stage 1: Reproject connectivity to EPSG:4326 ---
with rasterio.open(CONN_PATH) as conn_src:
    reproj_transform, reproj_width, reproj_height = calculate_default_transform(
        conn_src.crs, dst_crs,
        conn_src.width, conn_src.height,
        *conn_src.bounds
    )
    conn_reproj = np.zeros((reproj_height, reproj_width), dtype=np.float32)
    reproject(
        source=rasterio.band(conn_src, 1),
        destination=conn_reproj,
        src_transform=conn_src.transform,
        src_crs=conn_src.crs,
        dst_transform=reproj_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear
    )

print(f'Stage 1 — Reprojected connectivity shape: {conn_reproj.shape}  |  CRS: {dst_crs}')

# --- Stage 2: Resample to exact VNL grid ---
conn_aligned = np.zeros((vnl_height, vnl_width), dtype=np.float32)
reproject(
    source=conn_reproj,
    destination=conn_aligned,
    src_transform=reproj_transform,
    src_crs=dst_crs,
    dst_transform=vnl_transform,
    dst_crs=vnl_crs,
    resampling=Resampling.bilinear
)

print(f'Stage 2 — Aligned connectivity shape  : {conn_aligned.shape}')
print(f'          VNL shape                    : {vnl_raw.shape}')
assert conn_aligned.shape == vnl_raw.shape, 'ERROR: shape mismatch after alignment!'
print('\n✓ Both arrays share identical dimensions — grid alignment verified.')